# SME Data Warehouse
*Catalog/Schema fixed to `bh2026-winford-uc-dev.bh2026_sme_gold_lakehouse`*
*Each step below is isolated in its own cell to see results per query more clearly.*


## Data Flow
<img src="/Workspace/Users/@winforddavidyahooco.onmicrosoft.com/DE Workshop/bh2026_sme_gold_lakehouse/diagrams/data_flow_diagram.png" width="1500"/>

## Initialisation & Setup Bronze tables

## Bronze Layer
<img src="/Workspace/Users/@winforddavidyahooco.onmicrosoft.com/DE Workshop/bh2026_sme_gold_lakehouse/diagrams/imgs/Bronze Layer.jpg" width="1500"/>

In [0]:
-- Create Catalog & Schema (explicit)
-- CREATE CATALOG IF NOT EXISTS `bh2026-winford-uc-dev`;
USE CATALOG `bh2026-winford-uc-dev`;
CREATE SCHEMA IF NOT EXISTS bh2026_sme_gold_lakehouse;
USE SCHEMA bh2026_sme_gold_lakehouse;
SELECT current_catalog() AS current_catalog, current_schema() AS current_schema;

In [0]:
-- bronze_sales_raw: Raw sales transaction data.

CREATE TABLE IF NOT EXISTS bronze_sales_raw (
    transaction_id STRING,
    transaction_date STRING,
    customer_id STRING,
    product_id STRING,
    quantity STRING,
    price STRING,
    region_id STRING
);


-- bronze_products_raw: Raw product information.

CREATE TABLE IF NOT EXISTS bronze_products_raw (
    product_id STRING,
    product_name STRING,
    category STRING,
    unit_price STRING
);


-- bronze_customers_raw: Raw customer details.

CREATE TABLE IF NOT EXISTS bronze_customers_raw (
    customer_id STRING,
    customer_name STRING,
    email STRING,
    address STRING,
    city STRING,
    state STRING,
    zip_code STRING
);


-- bronze_regions_raw: Raw regional information.

CREATE TABLE IF NOT EXISTS bronze_regions_raw (
    region_id STRING,
    region_name STRING,
    country STRING
);



## Add Bronze tables data

In [0]:
INSERT INTO bronze_sales_raw VALUES
('T001', '2023-01-01', 'C001', 'P001', '2', '10.50', 'R001'),
('T002', '2023-01-01', 'C002', 'P002', '1', '25.00', 'R002'),
('T003', '2023-01-02', 'C001', 'P003', '3', '5.25', 'R001'),
('T004', '2023-02-01', 'C003', 'P001', '1', '10.50', 'R003'),
('T005', '2023-02-05', 'C002', 'P004', '2', '15.00', 'R002');
 
INSERT INTO bronze_products_raw VALUES
('P001', 'Laptop', 'Electronics', '10.50'),
('P002', 'Mouse', 'Electronics', '25.00'),
('P003', 'Keyboard', 'Electronics', '5.25'),
('P004', 'Monitor', 'Electronics', '15.00');
 
INSERT INTO bronze_customers_raw VALUES
('C001', 'Alice Smith', 'alice@example.com', '123 Main St', 'Anytown', 'CA', '90210'),
('C002', 'Bob Johnson', 'bob@example.com', '456 Oak Ave', 'Otherville', 'NY', '10001'),
('C003', 'Charlie Brown', 'charlie@example.com', '789 Pine Ln', 'Somewhere', 'TX', '75001');
 
INSERT INTO bronze_regions_raw VALUES
('R001', 'West', 'USA'),
('R002', 'East', 'USA'),
('R003', 'Central', 'USA');

## Silver Layer (Cleaned Data)

## Silver Layer
<img src="/Workspace/Users/@winforddavidyahooco.onmicrosoft.com/DE Workshop/bh2026_sme_gold_lakehouse/diagrams/imgs/Silver Layer Analysing and Understanding.jpg" width="1500"/>


The Silver layer refines the raw data from the Bronze layer. Here, data undergoes cleaning, standardization, and transformation processes. This layer is crucial for data quality and consistency, and it also implements **Slowly Changing Dimensions (SCD Type 2)** for tracking historical changes in dimension attributes.

**Conceptual Tables:**

•	silver_sales: Cleaned sales transactions, with appropriate data types and calculated fields.
\
•	silver_products: Standardized product information.
\
•	silver_customers: Cleaned customer data, **with SCD Type 2 attributes**.
\
•	silver_regions: Standardized regional data.


## SCD Type 2 init

In [0]:
CREATE TABLE IF NOT EXISTS silver_sales (
    transaction_id STRING,
    transaction_date DATE,
    customer_id STRING,
    product_id STRING,
    quantity INT,
    price DECIMAL(10, 2),
    total_amount DECIMAL(10, 2),
    region_id STRING
);
 
CREATE TABLE IF NOT EXISTS silver_products (
    product_id STRING,
    product_name STRING,
    category STRING,
    unit_price DECIMAL(10, 2)
);
 
CREATE TABLE IF NOT EXISTS silver_customers (
    customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id STRING,
    customer_name STRING,
    email STRING,
    address STRING,
    city STRING,
    state STRING,
    zip_code STRING,
    start_date DATE,
    end_date DATE,
    is_current BOOLEAN
);
 
CREATE TABLE IF NOT EXISTS silver_regions (
    region_id STRING,
    region_name STRING,
    country STRING
);


Insert data across the three MERGE statements (5 sales + 4 products + 3 regions affected).

Databricks use MERGE INTO statements matching on the respective business keys (transaction_id, product_id, region_id). 

As a result this is 'idempotent' whereby re-running will update existing rows rather than creating duplicates!

In [0]:
-- Ingest and transform sales data

MERGE INTO silver_sales AS target
USING (
    SELECT
        transaction_id,
        CAST(transaction_date AS DATE) AS transaction_date,
        customer_id,
        product_id,
        CAST(quantity AS INT) AS quantity,
        CAST(price AS DECIMAL(10, 2)) AS price,
        CAST(quantity AS INT) * CAST(price AS DECIMAL(10, 2)) AS total_amount,
        region_id
    FROM bronze_sales_raw
) AS source
ON target.transaction_id = source.transaction_id
WHEN MATCHED THEN UPDATE SET
    target.transaction_date = source.transaction_date,
    target.customer_id = source.customer_id,
    target.product_id = source.product_id,
    target.quantity = source.quantity,
    target.price = source.price,
    target.total_amount = source.total_amount,
    target.region_id = source.region_id
WHEN NOT MATCHED THEN INSERT (
    transaction_id,
    transaction_date,
    customer_id,
    product_id,
    quantity,
    price,
    total_amount,
    region_id
) VALUES (
    source.transaction_id,
    source.transaction_date,
    source.customer_id,
    source.product_id,
    source.quantity,
    source.price,
    source.total_amount,
    source.region_id
);
 
-- Ingest and transform product data
MERGE INTO silver_products AS target
USING (
    SELECT
        product_id,
        product_name,
        category,
        CAST(unit_price AS DECIMAL(10, 2)) AS unit_price
    FROM bronze_products_raw
) AS source
ON target.product_id = source.product_id
WHEN MATCHED THEN UPDATE SET
    target.product_name = source.product_name,
    target.category = source.category,
    target.unit_price = source.unit_price
WHEN NOT MATCHED THEN INSERT (
    product_id,
    product_name,
    category,
    unit_price
) VALUES (
    source.product_id,
    source.product_name,
    source.category,
    source.unit_price
);
 
-- Ingest and transform region data
MERGE INTO silver_regions AS target
USING (
    SELECT
        region_id,
        region_name,
        country
    FROM bronze_regions_raw
) AS source
ON target.region_id = source.region_id
WHEN MATCHED THEN UPDATE SET
    target.region_name = source.region_name,
    target.country = source.country
WHEN NOT MATCHED THEN INSERT (
    region_id,
    region_name,
    country
) VALUES (
    source.region_id,
    source.region_name,
    source.country
);

## SCD Type 2 Preserve Historical Customer data


In [0]:
-- SCD Type 2 for Customers
MERGE INTO silver_customers AS target
USING (
    SELECT
        customer_id,
        customer_name,
        email,
        address,
        city,
        state,
        zip_code
    FROM bronze_customers_raw
) AS source
ON target.customer_id = source.customer_id AND target.is_current = TRUE
WHEN MATCHED AND (
    target.customer_name <> source.customer_name OR
    target.email <> source.email OR
    target.address <> source.address OR
    target.city <> source.city OR
    target.state <> source.state OR
    target.zip_code <> source.zip_code
) THEN UPDATE SET
    target.is_current = FALSE,
    target.end_date = CURRENT_DATE()
WHEN NOT MATCHED THEN INSERT (
    customer_id,
    customer_name,
    email,
    address,
    city,
    state,
    zip_code,
    start_date,
    end_date,
    is_current
) VALUES (
    source.customer_id,
    source.customer_name,
    source.email,
    source.address,
    source.city,
    source.state,
    source.zip_code,
    CURRENT_DATE(),
    NULL,
    TRUE
);


My understanding of SCD Dimension Type 2 preserves the full history of changes to dimension attributes by creating a new row each time a tracked attribute changes, rather than overwriting the old value. 
This is critical for analytics that need to answer "what did this customer look like at the time of a transaction?"

Table Schema Design, my 'silver_customers' table has three SCD Type 2 control columns beyond the business data:

**Column - Purpose:**
customer_sk - Surrogate key (auto-generated identity) — uniquely identifies each version of a customer
\
**Column - Purpose:**
start_date - When this version became effective
\
**Column - Purpose**:
end_date - When this version was superseded (NULL = still active)
\
**Column - Purpose**:
is_current - Boolean flag for easy filtering to the latest version

The customer_id is the business key (natural key) — it stays the same across versions. The surrogate key (customer_sk) distinguishes each historical snapshot.

So, the way the 'MERGE' Works:
Match condition: target.customer_id = source.customer_id AND target.is_current = TRUE — only compares incoming data against the current active row for each customer.

WHEN MATCHED AND attributes differ — If Alice moves from "123 Main St" to "456 Elm St", the MERGE in this case:

Sets is_current = FALSE on the existing row
Sets end_date = CURRENT_DATE() — closing out that version

WHEN NOT MATCHED — Brand-new customers get inserted with start_date = today, end_date = NULL, is_current = TRUE.
In my first time running the Cell, it created/inserted 6 x new rows, as there were no previous customers, matching any of the requirements.

Important Nuance
This MERGE handles two of three SCD2 scenarios in one pass. However, it expires the old row but doesn't insert the new version in the same statement. A second run of the same MERGE will then pick up the changed customer as "NOT MATCHED" (since no is_current = TRUE row exists) and insert the new version. This is a common two-pass pattern — idempotent and safe to re-run, which aligns with my orignal pipeline design principles.

Business Value
Without SCD2, if I try to JOIN fact_sales to dim_customers, a customer who moved from CA to NY would show all historical sales as NY. With SCD2, I can JOIN on customer_sk (or date-range logic) to see that orders placed in January were shipped to CA, and orders in March went to NY — which is essential for accurate regional revenue reporting and cohort analysis.

## Gold Layer (Curated for Business Intelligence)

## Gold Layer
<img src="/Workspace/Users/@winforddavidyahooco.onmicrosoft.com/DE Workshop/bh2026_sme_gold_lakehouse/diagrams/imgs/Gold Layer.jpg" width="1500"/>

## Table Definitions
<img src="/Workspace/Users/@winforddavidyahooco.onmicrosoft.com/DE Workshop/bh2026_sme_gold_lakehouse/diagrams/Table Definitions.jpg" width="1500"/>

## Star Schema
<img src="/Workspace/Users/@winforddavidyahooco.onmicrosoft.com/DE Workshop/bh2026_sme_gold_lakehouse/diagrams/star_schema.png" width="1500"/>

## Github Repositry
<img src="/Workspace/Users/@winforddavidyahooco.onmicrosoft.com/DE Workshop/bh2026_sme_gold_lakehouse/diagrams/imgs/github_readme.jpg" width="1500"/>


# GOLD CODE !!!